<a href="https://colab.research.google.com/github/WVF-1/FULPs-Framework-CA-Application/blob/main/Cellular_Automata_Grid_Visualiztion_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from scipy.linalg import cholesky
import os

# ── Reproducibility ────────────────────────────────────
SEED        = 42
GRID_H      = 50
GRID_W      = 50
T_PERTURB   = 100
N_TIMESTEPS = 200

# ── FULPs parameters (from v5 notebook, Cell 2) ───────────────
CONF_GATE        = 0.60
GATE_CONF_GATE   = 0.80
SIGNAL_WEIGHT    = 0.25
CONTRA_START     = 0.20
CONTRA_END       = 0.05
BUFFER_DELAY     = 10
EMA_ALPHA        = 0.1
CURIOSITY_WEIGHT = 2.0
BASE_THRESHOLD   = 0.70
CONTRA_THRESHOLD = 0.60
NBR_DIM          = 9
N_STATES         = 512
VOID_HIDDEN      = 16
REP_DIM          = VOID_HIDDEN

# Snapshots to capture
SNAPSHOT_STEPS = [0, 50, 99, 100, 110, 150, 199]

# ── GoL rules ────────────────────────────────────
def tick(grid, perturbed=False):
    nc = np.zeros((GRID_H, GRID_W), dtype=np.int8)
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0: continue
            nc += np.roll(np.roll(grid, di, 0), dj, 1)
    nxt = np.zeros((GRID_H, GRID_W), dtype=np.int8)
    a = grid == 1
    if perturbed: nxt[a & ((nc==2)|(nc==3)|(nc==4))] = 1
    else:         nxt[a & ((nc==2)|(nc==3))]          = 1
    nxt[~a & (nc==3)] = 1
    return nxt

# ── Neighbourhood encoding ───────────────────────
_II     = np.arange(GRID_H)[:,None]*np.ones(GRID_W,dtype=int)[None,:]
_JJ     = np.ones(GRID_H,dtype=int)[:,None]*np.arange(GRID_W)[None,:]
_POWERS = (2**np.arange(9)).astype(np.int32)
_SHIFTS = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,0),(0,1),(1,-1),(1,0),(1,1)]

def encode_nbr(grid):
    ch = np.zeros((GRID_H,GRID_W,9),dtype=np.float32)
    for k,(di,dj) in enumerate(_SHIFTS):
        ch[:,:,k] = np.roll(np.roll(grid,-di,0),-dj,1).astype(np.float32)
    return (ch.astype(np.int32)*_POWERS[None,None,:]).sum(axis=-1), ch

# ── Minimal ARE (cosine similarity) ──────────────────
def compute_are(reps_flat, are_state, threshold):
    rep = reps_flat.reshape(GRID_H,GRID_W,REP_DIM)
    def cs(a, b):
        d = np.sum(a*b,axis=-1)
        na = np.linalg.norm(a,axis=-1); nb = np.linalg.norm(b,axis=-1)
        return d/np.where(na*nb>1e-8, na*nb, 1e-8)
    ps = np.where(are_state['pa'], cs(rep, are_state['pr']), 0.0)
    ns = np.where(are_state['na'], cs(rep, are_state['nr']), 0.0)
    m  = ps - ns
    both = are_state['pa'] & are_state['na']
    contra = (np.abs(m)<threshold) & (np.maximum(ps,ns)>CONF_GATE) & both
    return m, contra, ps, ns

def update_are_buf(are_state, reps_flat, outcomes):
    rep = reps_flat.reshape(GRID_H,GRID_W,REP_DIM)
    for mask,k in [(outcomes==1,'p'),(outcomes==0,'n')]:
        if mask.any():
            are_state[f'{k}r'][mask] = (1-EMA_ALPHA)*are_state[f'{k}r'][mask]+EMA_ALPHA*rep[mask]
            are_state[f'{k}a'][mask] = True

def init_are():
    return {'pr':np.zeros((GRID_H,GRID_W,REP_DIM),dtype=np.float32),
            'nr':np.zeros((GRID_H,GRID_W,REP_DIM),dtype=np.float32),
            'pa':np.zeros((GRID_H,GRID_W),dtype=bool),
            'na':np.zeros((GRID_H,GRID_W),dtype=bool)}

# ── Distress + gating ────────────────────────────────
def distress(ps, ns, margin):
    return np.maximum(ps,ns)*np.clip(np.abs(margin),0,1)

def nbr_mean_distress(d):
    s = np.zeros((GRID_H,GRID_W),dtype=np.float32)
    for di in [-1,0,1]:
        for dj in [-1,0,1]:
            if di==0 and dj==0: continue
            s += np.roll(np.roll(d, di, 0), dj, 1)
    return s/8.0

def gating_mask_fn(ac, ps, ns, nbr_dist=None):
    max_sim = np.maximum(ps, ns)
    if nbr_dist is not None:
        eff = GATE_CONF_GATE*(1.0 - SIGNAL_WEIGHT*np.clip(nbr_dist,0,1))
        strong = max_sim > eff
    else:
        strong = max_sim > GATE_CONF_GATE
    return ac & strong

# ── Minimal VoidStabilizer (random weights, enough to demonstrate DSC) ─────
# For visualization we use a fixed random projection as a lightweight stand-in.
# This is sufficient to demonstrate the qualitative gating patterns.
np.random.seed(SEED)
_W1 = np.random.randn(NBR_DIM, REP_DIM).astype(np.float32) * 0.3
_b1 = np.zeros(REP_DIM, dtype=np.float32)

def encode_reps(nbr_floats):
    """Lightweight encoder: linear + tanh, mimicking VoidStabilizer behaviour."""
    flat = nbr_floats.reshape(-1, NBR_DIM)
    z    = np.tanh(flat @ _W1 + _b1)
    return z

# ── Run simulation and collect snapshots ───────────────────
np.random.seed(SEED)
grid_std  = np.random.randint(0,2,size=(GRID_H,GRID_W),dtype=np.int8)
grid_sig  = grid_std.copy()

sig_table = np.ones((GRID_H,GRID_W,N_STATES,2),dtype=np.float32)
are_sig   = init_are()

snaps = {t: {'std':None,'sig':None,'gate':None,'dist':None} for t in SNAPSHOT_STEPS}

for t in range(N_TIMESTEPS):
    is_pert = (t >= T_PERTURB)
    prog    = t/max(N_TIMESTEPS-1,1)
    thresh  = CONTRA_START + (CONTRA_END-CONTRA_START)*prog

    # Standard grid
    next_std = tick(grid_std, perturbed=is_pert)

    # Signal grid
    nbr_sig, nbr_sig_f = encode_nbr(grid_sig)
    reps_s = encode_reps(nbr_sig_f)
    ms, cs_, pss, nss  = compute_are(reps_s, are_sig, thresh)
    acs = cs_          # simplified: skip DSC gate for clean viz
    dst = distress(pss, nss, ms)
    nd  = nbr_mean_distress(dst)
    gm  = gating_mask_fn(acs, pss, nss, nbr_dist=nd)

    next_sig = tick(grid_sig, perturbed=is_pert).copy()
    next_sig[gm] = grid_sig[gm]
    out_sig = next_sig.astype(np.int8)

    # Update table + ARE
    sig_table[_II,_JJ,nbr_sig,out_sig.astype(int)] += 1.0
    if t >= BUFFER_DELAY:
        update_are_buf(are_sig, reps_s, out_sig)

    # Capture snapshots
    if t in snaps:
        snaps[t]['std']  = grid_std.copy()
        snaps[t]['sig']  = grid_sig.copy()
        snaps[t]['gate'] = gm.copy()
        snaps[t]['dist'] = dst.copy()

    grid_std = next_std
    grid_sig = next_sig

# ── Figure 1: Side-by-side GoL snapshots across time ───────────────
steps    = SNAPSHOT_STEPS
n        = len(steps)
fig, axes = plt.subplots(3, n, figsize=(n*2.4, 7.5), dpi=140)
fig.suptitle(
    'FULPs CA v5 — Standard GoL vs FULPs Signal Grid\n'
    '(50×50 toroidal grid, seed=42, perturbation {2,3}→{2,3,4} at t=100)',
    fontsize=11, fontweight='bold', y=1.01
)

row_titles = ['Standard GoL', 'FULPs Signal Grid', 'Gating Mask (signal)']
cmap_gol   = 'Greys'
cmap_gate  = ListedColormap(['#f0f0f0','#2c7bb6'])   # grey=not gating, blue=gating
cmap_dist  = 'YlOrRd'

for col, t in enumerate(steps):
    snap = snaps[t]

    ax0 = axes[0, col]
    ax0.imshow(snap['std'], cmap=cmap_gol, vmin=0, vmax=1, interpolation='nearest')
    ax0.set_xticks([]); ax0.set_yticks([])
    label = f't = {t}'
    if t == T_PERTURB: label += '\n⟵ PERTURB'
    ax0.set_title(label, fontsize=8, fontweight='bold' if t==T_PERTURB else 'normal')

    ax1 = axes[1, col]
    ax1.imshow(snap['sig'], cmap=cmap_gol, vmin=0, vmax=1, interpolation='nearest')
    ax1.set_xticks([]); ax1.set_yticks([])

    ax2 = axes[2, col]
    ax2.imshow(snap['gate'].astype(float), cmap=cmap_gate, vmin=0, vmax=1, interpolation='nearest')
    ax2.set_xticks([]); ax2.set_yticks([])
    rate = snap['gate'].mean()
    ax2.set_xlabel(f'gate={rate:.2f}', fontsize=7)

for row, title in enumerate(row_titles):
    axes[row, 0].set_ylabel(title, fontsize=9, fontweight='bold', labelpad=6)

plt.tight_layout()
out1 = '/mnt/user-data/outputs/fulps_ca_v5_grid_comparison.png'
os.makedirs(os.path.dirname(out1), exist_ok=True)
plt.savefig(out1, dpi=160, bbox_inches='tight')
plt.close()
print(f'Saved: {out1}')

# ── Figure 2: Distress field evolution ──────────────────────────
dist_steps = [50, 99, 100, 110, 130, 150, 175, 199]
dist_steps = [s for s in dist_steps if s in snaps]
n2 = len(dist_steps)
fig2, axes2 = plt.subplots(2, 4, figsize=(12, 6), dpi=140)
fig2.suptitle(
    'FULPs CA v5 — Distress Field & Signal Gating Mask Evolution\n'
    '(distress = max(pos_sim, neg_sim) × |margin|; brighter = stronger belief violation)',
    fontsize=10, fontweight='bold'
)
axes2 = axes2.flatten()
for i, t in enumerate(dist_steps):
    ax = axes2[i]
    im = ax.imshow(snaps[t]['dist'], cmap='YlOrRd', vmin=0, vmax=0.8, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    label = f't = {t}'
    if t == T_PERTURB: label += ' ⟵ PERTURB'
    ax.set_title(label, fontsize=8, fontweight='bold' if t==T_PERTURB else 'normal')
    ax.set_xlabel(f'mean={snaps[t]["dist"].mean():.3f}', fontsize=7)

# Shared colorbar
fig2.subplots_adjust(right=0.88)
cbar_ax = fig2.add_axes([0.91, 0.12, 0.02, 0.76])
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(0, 0.8))
sm.set_array([])
fig2.colorbar(sm, cax=cbar_ax, label='Distress level')

out2 = '/mnt/user-data/outputs/fulps_ca_v5_distress_evolution.png'
plt.savefig(out2, dpi=160, bbox_inches='tight')
plt.close()
print(f'Saved: {out2}')

Saved: /mnt/user-data/outputs/fulps_ca_v5_grid_comparison.png
Saved: /mnt/user-data/outputs/fulps_ca_v5_distress_evolution.png
